# Structured Output

Um modelo de linguagem produz texto. Uma aplicação precisa de valores: um rótulo de um conjunto fechado, um booleano, um inteiro dentro de uma faixa. Saída estruturada é o conjunto de técnicas que atravessa essa distância, e cada uma atua em um momento diferente da chamada, do pedido escrito no prompt até a restrição imposta na escolha do próximo token.

A tarefa é extrair três campos de uma resenha de produto. O caminho começa pedindo JSON e vendo o que volta, passa por um parser tolerante, por um esquema que recusa o que não obedece ao contrato, por uma nova tentativa com o erro devolvido ao modelo, e termina restringindo a geração para que a saída inválida deixe de ser possível.

In [ ]:
import json

import pandas as pd
import torch
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

from agentkit import LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Do texto ao valor

Os três campos a extrair têm tipos diferentes entre si, o que faz aparecerem falhas diferentes: o sentimento vem de um conjunto fechado, a recomendação é um booleano e a nota é um inteiro em uma faixa.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=160)
print(llm.model)

In [ ]:
review = (
    "I bought this backpack for a long trip and used it every day. "
    "Nothing broke and it still looks new. I give it 9 out of 10 and I would buy it again."
)
target = {"sentiment": "positive", "recommends": True, "rating": 9}
target

O primeiro pedido descreve os campos em uma frase e pede JSON.

In [ ]:
raw = llm.invoke([
    {"role": "user", "content": (
        "Extract the sentiment, whether the customer recommends the product, "
        f"and the rating from this review as JSON:\n\n{review}"
    )},
])
print(raw)

O conteúdo está correto. A célula seguinte tenta usá-lo como um programa usaria.

In [ ]:
json.loads(raw)

A chamada falha na primeira coluna, porque a resposta começa com a cerca de bloco de código em vez da chave. Três problemas distintos convivem nessa saída: a moldura em volta do JSON, o nome do campo, que veio como `recommendation` e não como o nome pedido, e o tipo do valor, que em outras execuções aparece como texto no lugar de booleano.

Cada um é resolvido por um mecanismo diferente, e as próximas partes tratam de um por vez.

## Extração tolerante

O primeiro problema é de moldura. O texto útil está entre a primeira chave e a última, e recortá-lo resolve a cerca, o comentário antes do objeto e a frase de cortesia depois.

In [ ]:
def extract_json(text: str) -> dict | None:
    """Recorta o primeiro objeto JSON do texto e devolve None quando não há um válido."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return None

In [ ]:
data = extract_json(raw)
data

O recorte funcionou e o dicionário existe, com um campo chamado `recommendation` que o código que consome esse resultado não espera. O parser resolveu a moldura e não tem como resolver o contrato, porque ele não sabe quais campos deveriam estar ali.

A decisão de projeto aqui é devolver `None` em vez de levantar exceção, porque a saída malformada é um caso normal desta tarefa e o chamador precisa decidir o que fazer com ela.

## Validação com esquema

O contrato precisa existir em algum lugar do código. Escrito como uma sequência de `if` sobre o dicionário, ele se espalha e diverge da documentação na primeira alteração. Escrito como declaração de tipos, ele serve ao mesmo tempo de validação, de documentação e de fonte da mensagem de erro.

### Pydantic

O Pydantic é a biblioteca padrão de Python para essa declaração. Uma classe herda de `BaseModel` e descreve os campos com anotações de tipo; `Field` acrescenta restrições de faixa; `Literal` fecha o conjunto de valores aceitos. A validação acontece em tempo de execução, o que é o que interessa em uma fronteira por onde entra texto gerado.

In [ ]:
class Review(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    recommends: bool
    rating: int = Field(ge=0, le=10)

In [ ]:
parsed = Review.model_validate(target)
parsed

O retorno é um objeto, e não o dicionário original. Os campos passam a ser acessíveis por atributo, com tipo garantido, e o caminho de volta ao dicionário continua disponível.

In [ ]:
print(parsed.rating + 1)
print(parsed.model_dump())

### Erros de validação

O dicionário que veio do modelo não passa. O erro é capturado para ser lido, porque ele volta a ser usado adiante.

In [ ]:
try:
    Review.model_validate(data)
except ValidationError as error:
    print(error)

A mensagem nomeia o campo que falta e o tipo do erro, em texto pronto para entrar em um prompt. Essa propriedade é o que torna a validação útil além de barrar o dado ruim.

O campo inventado passou sem reclamação, porque o comportamento padrão é ignorar chaves fora do esquema. Declarar `model_config = ConfigDict(extra="forbid")` faria a validação recusá-lo, e a escolha entre as duas depende de quem produz o dado.

### Esquema declarado no prompt

O erro aponta a causa: o prompt descrevia a tarefa em prosa e nunca declarou os nomes dos campos. A instrução abaixo declara o contrato em texto.

In [ ]:
INSTRUCTION = """Extract these fields from the review and reply with JSON only.
sentiment: one of "positive", "neutral", "negative"
recommends: true or false
rating: integer from 0 to 10"""
print(INSTRUCTION)

Essa instrução repete, em prosa, o que a classe já diz. Duas escritas do mesmo contrato divergem na primeira alteração, e o Pydantic evita isso gerando o esquema a partir da própria classe.

In [ ]:
print(json.dumps(Review.model_json_schema(), indent=2))

In [ ]:
SCHEMA_INSTRUCTION = (
    "Extract the fields from the review and reply with JSON only, "
    "matching this JSON Schema:\n" + json.dumps(Review.model_json_schema())
)
print(SCHEMA_INSTRUCTION)

A comparação entre as três versões precisa de mais de uma execução, porque a falha é intermitente. As amostras usam temperatura acima de zero, que é a condição em que o problema aparece na prática.

In [ ]:
def validity_rate(prompt: str, samples: int = 4) -> float:
    """Mede a fração de amostras que passam na validação do esquema."""
    valid = 0
    for seed in range(samples):
        torch.manual_seed(seed)
        answer = llm.invoke([{"role": "user", "content": prompt}], temperature=0.7)
        try:
            Review.model_validate(extract_json(answer))
            valid += 1
        except ValidationError:
            pass
    return valid / samples

In [ ]:
loose = f"Extract sentiment, recommends and rating from this review as JSON:\n\n{review}"
strict = f"{INSTRUCTION}\n\nReview: {review}"
generated = f"{SCHEMA_INSTRUCTION}\n\nReview: {review}"
pd.DataFrame(
    {
        "prompt": ["descrição em prosa", "esquema em prosa", "esquema gerado pela classe"],
        "tokens": [len(llm.tokenizer.encode(prompt)) for prompt in [loose, strict, generated]],
        "validity": [validity_rate(loose), validity_rate(strict), validity_rate(generated)],
    }
)

Declarar o contrato no prompt é a intervenção mais barata e a que mais muda o número. O esquema gerado custa mais tokens que a versão em prosa e tem a vantagem de acompanhar a classe sozinho. As duas continuam sendo um pedido, e as próximas partes tratam do que sobra.

### Nova tentativa com o erro

Declarar o contrato resolve a resenha fácil e não resolve todas. A resenha abaixo é difícil de propósito: mistura elogio e reclamação, e cita uma nota fora da escala. O modelo tende a inventar um sentimento fora do conjunto e a devolver nota fora da faixa permitida.

In [ ]:
hard_review = (
    "Honestly it is a mixed bag: great fabric, terrible zipper. "
    "I would give it 12 out of 10 for looks and 2 for durability."
)
hard_prompt = f"{INSTRUCTION}\n\nReview: {hard_review}"

In [ ]:
for seed in range(4):
    torch.manual_seed(seed)
    print(extract_json(llm.invoke([{"role": "user", "content": hard_prompt}], temperature=0.7)))

Parte das saídas viola o contrato, e a mensagem de validação diz exatamente em quê. Devolver essa mensagem ao modelo transforma a validação em correção.

In [ ]:
def generate_valid(prompt: str, schema: type, attempts: int = 3):
    """Repete a chamada devolvendo o erro de validação ao modelo até obter um objeto válido."""
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(1, attempts + 1):
        answer = llm.invoke(messages, temperature=0.7)
        try:
            return schema.model_validate(extract_json(answer)), attempt
        except ValidationError as error:
            messages = [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer},
                {"role": "user", "content": f"That answer is invalid:\n{error}\n\nReply again with corrected JSON only."},
            ]
    return None, attempts

In [ ]:
rows = []
for seed in range(3):
    torch.manual_seed(seed)
    result, attempts = generate_valid(hard_prompt, Review)
    rows.append({"seed": seed, "attempts": attempts, "result": result})
pd.DataFrame(rows)

As três execuções mostram os três desfechos possíveis: acerto na primeira chamada, recuperação na segunda e esgotamento do limite. O laço aumenta a taxa de validade e não a garante, e o limite existe porque a repetição pode não convergir. Esse é o único mecanismo disponível para regras que o esquema não expressa, como exigir que a nota seja a que o texto cita, e por isso ele sobrevive ao que vem na parte seguinte.

## Geração guiada por esquema

Todas as técnicas anteriores pedem ao modelo que obedeça. A alternativa é remover a possibilidade de desobedecer, restringindo o conjunto de tokens que podem ser escolhidos em cada posição.

### Escolha restrita entre rótulos

O caso mais simples é a classificação, em que a resposta inteira é um rótulo de um conjunto conhecido. Em vez de gerar texto livre e conferir depois, o forward pass é executado uma vez e os logits dos rótulos candidatos são comparados entre si.

In [ ]:
LABELS = ["positive", "neutral", "negative"]
label_ids = [llm.tokenizer.encode(label)[0] for label in LABELS]
pd.DataFrame({"label": LABELS, "first_token_id": label_ids})

Os três rótulos começam em tokens diferentes, o que permite decidir com uma única posição.

In [ ]:
def classify(text: str) -> str:
    """Escolhe o rótulo de maior logit na primeira posição da resposta."""
    messages = [
        {"role": "system", "content": "Classify the sentiment of the review."},
        {"role": "user", "content": text},
    ]
    prompt = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm.tokenizer(prompt, return_tensors="pt").to(llm.weights.device)
    with torch.no_grad():
        logits = llm.weights(**inputs).logits[0, -1]
    return LABELS[int(logits[label_ids].argmax())]

In [ ]:
samples = [
    review,
    "The zipper broke after two days and support never replied.",
    "It works, nothing special.",
]
pd.DataFrame({"label": [classify(text) for text in samples], "review": samples})

A saída inválida deixou de existir, porque o conjunto de respostas possíveis é o conjunto de rótulos. A validade passou a ser propriedade do método e não do comportamento do modelo, e nenhuma nova tentativa é necessária.

O método decide com o primeiro token. Rótulos que compartilhassem esse token exigiriam comparar a sequência inteira, e um objeto com vários campos exigiria refazer a restrição a cada posição, porque o que é permitido depende do que já foi escrito.

### Outlines

A biblioteca Outlines faz exatamente isso. Ela deriva do esquema um autômato que descreve todas as continuações válidas, e a cada passo zera a probabilidade dos tokens que levariam a uma saída fora do esquema. O modelo continua escolhendo, dentro do que restou.

Outlines é dependência opcional do `agentkit` e entra com `pip install -e ".[structured]"`. A construção abaixo reaproveita os pesos e o tokenizador já carregados, sem baixar nada de novo.

In [ ]:
from outlines import Generator, from_transformers

guided_model = from_transformers(llm.weights, llm.tokenizer)
generator = Generator(guided_model, Review)

In [ ]:
hard_messages = [{"role": "user", "content": f"Extract sentiment, recommends and rating.\n\nReview: {hard_review}"}]
guided_prompt = llm.tokenizer.apply_chat_template(
    hard_messages, tokenize=False, add_generation_prompt=True
)
print(generator(guided_prompt, max_new_tokens=80, do_sample=False))

A saída é uma string, sempre analisável e sempre dentro do esquema, mesmo com a instrução em prosa que falhava antes. A cerca de bloco de código e o campo com nome errado deixaram de ser possíveis, porque os tokens que os iniciariam foram zerados.

### generate_structured

A classe `LLM` encapsula essa rota em um método, para que o restante do código receba o objeto validado em vez da string.

In [ ]:
extracted = llm.generate_structured(hard_messages, Review, max_tokens=80)
print(extracted, type(extracted).__name__)

In [ ]:
print(extracted.rating, extracted.sentiment)
print(llm.last_usage)

Por dentro o método faz quatro coisas: aplica o template de conversa quando recebe mensagens, monta o gerador guiado com os pesos já carregados, gera com amostragem desligada e devolve `schema.model_validate_json` sobre o texto produzido. O registro de uso continua sendo preenchido, o que mantém a comparação de custo com as outras rotas.

A garantia é de forma. A nota devolvida está dentro da faixa porque o esquema a impõe, e o valor escolhido continua sendo uma leitura do modelo sobre uma resenha que dizia doze de dez. Validade e verdade são propriedades separadas, e só a segunda exige um conjunto de avaliação com rótulos conferidos à mão.

### Casos de uso

O esquema é o que define a tarefa, e trocar a classe troca o problema resolvido sem mudar nada no laço de chamada. Os três casos abaixo cobrem as formas que mais aparecem, da mais simples para a mais composta.

O primeiro é o campo que pode não existir no texto. A anotação `str | None` autoriza a ausência.

In [ ]:
class Contact(BaseModel):
    name: str
    email: str | None
    phone: str | None

In [ ]:
contact_text = "Hi, this is Paulo from the operations team, you can reach me at paulo@acme.com."
llm.generate_structured(
    [{"role": "user", "content": f"Extract the contact.\n\n{contact_text}"}], Contact, max_tokens=120
)

O telefone não está no texto e o campo veio preenchido com uma frase. O esquema admite string ou nulo, e nada obriga o modelo a preferir o nulo quando o dado falta. Autorizar a ausência é diferente de exigi-la: ou a instrução diz o que fazer com o campo faltante, ou o tipo precisa recusar a saída indesejada, com um `Literal` que não inclua texto livre.

O segundo caso trata vários registros em uma chamada, com a lista no topo do esquema.

In [ ]:
class Triage(BaseModel):
    index: int
    label: Literal["billing", "technical", "account"]


class Batch(BaseModel):
    results: list[Triage]

In [ ]:
batch_tickets = [
    "My card was charged twice.",
    "The app crashes on startup.",
    "I want to delete my profile.",
]
llm.generate_structured(
    [{"role": "user", "content": "Classify each ticket.\n\n" + "\n".join(batch_tickets)}],
    Batch,
    max_tokens=200,
)

In [ ]:
print(llm.last_usage)

Os três chamados foram resolvidos em uma chamada, com um prompt só e uma passagem de contexto só, o que reduz o custo em relação a três chamadas separadas. Em troca, o alinhamento entre entrada e saída passa a depender do modelo: o índice devolvido é um campo como outro qualquer, e o esquema não garante que ele aponte para o chamado certo. Lotes grandes também competem com a janela e degradam a qualidade das últimas linhas.

O terceiro caso é o mais composto, com um esquema dentro do outro. O campo `items` recebe uma lista de `Item`, e cada elemento dessa lista é validado com as regras da classe que o define.

In [ ]:
class Item(BaseModel):
    name: str
    quantity: int = Field(ge=1)
    unit_price: float


class Order(BaseModel):
    customer: str
    items: list[Item]
    urgent: bool

In [ ]:
order_text = (
    "Order from Maria Lima: 3 mechanical keyboards at 89.90 each and 1 monitor at 1299.00. "
    "She needs everything before Friday."
)
order = llm.generate_structured(
    [{"role": "user", "content": f"Extract the order.\n\n{order_text}"}], Order, max_tokens=200
)
order

In [ ]:
print(order.items[0].name, order.items[0].quantity)
print(sum(item.quantity * item.unit_price for item in order.items))

A lista chegou tipada, e o total sai de uma soma comum sobre objetos. O prazo mencionado no texto virou `urgent=True`, que é uma leitura do modelo e não um dado copiado.

O aninhamento é a forma geral: cada nível traz as próprias restrições, e a máscara de geração as respeita todas ao mesmo tempo, porque o autômato foi construído a partir do esquema inteiro. O custo aparece no número de tokens, que cresce com a profundidade da estrutura, e no risco de a estrutura ficar grande demais para o orçamento de geração.

## Exercícios

### Exercício 1

Cinco mensagens chegaram na fila de suporte. Descreva um contrato com categoria, urgência e um resumo de uma linha, e extraia as cinco para um `DataFrame`. As categorias possíveis são `billing`, `technical`, `account` e `feature_request`, e a urgência é um inteiro de 1 a 5, em que 5 significa serviço indisponível para muita gente. Aponte a mensagem cuja urgência ficou incompatível com o texto.

In [ ]:
TICKETS = [
    "The payment page returns error 500 and no customer can check out.",
    "I would like to know if the annual plan has a discount.",
    "My card was charged three times for the same order this morning.",
    "It would be great if the report could be exported to Excel.",
    "Nobody from my team can log in since the last update.",
]

### Exercício 2

As cinco resenhas abaixo falam de aspectos diferentes do mesmo produto, com opiniões que variam dentro do mesmo texto. Descreva um contrato que devolva uma lista de aspectos, cada um com nome e sentimento, e extraia as cinco. Diga o que veio na lista da resenha que não menciona aspecto nenhum.

In [ ]:
ASPECT_REVIEWS = [
    "The fabric is excellent and the zipper broke in a week.",
    "Delivery took eleven days, which is unacceptable, but support answered fast.",
    "Great price for what it offers, although the color is nothing like the photos.",
    "I have no complaints so far.",
    "The app is fast, the battery drains quickly and the manual is useless.",
]

### Exercício 3

Transforme pedidos em linguagem natural em comandos de agenda. O comando é um de `create`, `cancel`, `reschedule` ou `list`, e o contrato tem ainda pessoa, data e hora. Extraia os cinco pedidos e responda o que o modelo colocou nos campos de data e hora dos pedidos que não trazem esses dados.

In [ ]:
REQUESTS = [
    "Marque uma reunião com a Ana na quinta às 15h.",
    "Cancela o alinhamento com o time de dados de amanhã cedo.",
    "Preciso remarcar a conversa com o cliente para a semana que vem.",
    "Quais são os meus compromissos de sexta?",
    "Agenda trinta minutos com o Bruno hoje depois do almoço.",
]

### Exercício 4

Da conversa abaixo, extraia nome, idade, cidade, profissão e preferência de contato da pessoa. Compare o resultado com o texto e aponte o campo que o modelo preencheu sem ter lido.

In [ ]:
CONVERSATION = [
    {"role": "user", "content": "Oi, meu nome é Rafael Andrade, tenho 34 anos e moro em Natal."},
    {"role": "assistant", "content": "Prazer, Rafael. Como posso ajudar?"},
    {"role": "user", "content": "Trabalho como analista de logística. Prefiro que me avisem por e-mail, nunca por telefone."},
]

### Exercício 5

Os relatos abaixo são anotações de triagem. Descreva um contrato com nome, idade, sintoma principal, há quantos dias os sintomas começaram e a gravidade, que é `mild`, `moderate` ou `severe`, e extraia os cinco para um `DataFrame`. Responda o que aconteceu no relato em que a duração não aparece de forma numérica.

In [ ]:
PATIENTS = [
    "Ana Beatriz, 27, com dor de garganta e febre baixa há três dias.",
    "Paciente Carlos Menezes, 61 anos, falta de ar intensa desde ontem à noite.",
    "Juliana Rocha, 45, dor de cabeça leve que vai e volta há mais ou menos duas semanas.",
    "Menino de 8 anos, Pedro Lima, tosse seca persistente há cinco dias, sem febre.",
    "Marta Souza, 52, dor lombar forte que começou logo depois do carnaval.",
]